# Module 03 — Du CSV à Parquet

**Formation Big Data — ANSD / Data Innovation Lab**

Nous avons changé de moteur trois fois. Changeons maintenant de **carburant**.

Au programme :

1. convertir, et comparer tailles et temps de lecture ;
2. choisir un codec de compression ;
3. la **projection de colonnes** — le gain le plus spectaculaire ;
4. l'anatomie d'un fichier Parquet : groupes de lignes et statistiques ;
5. le **saut de blocs**, et la condition pour qu'il fonctionne.

> Contrairement aux blocs précédents, ce que nous gagnons ici profite aux
> **quatre outils à la fois**. Changer de moteur avantage un outil ; changer de
> format avantage tout le monde.

## 1. Préparation

In [1]:
%load_ext autoreload 
%autoreload 2
import sys
from pathlib import Path
try:
    sys.path.append(str(Path(__file__).parent.parent.resolve()))
except NameError:
    sys.path.append(str(Path.cwd().parent.resolve()))
import gc
import time

import pandas as pd
import pyarrow.parquet as pq

from tools.outils_mesure import FICHIER, contexte_machine, mesurer

DOSSIER_DONNEES = FICHIER.parent
PARQUET = DOSSIER_DONNEES / "individus.parquet"

contexte_machine()

Cœurs logiques    : 12
Cœurs physiques   : 6
Mémoire totale    : 16.0 Go
Mémoire libre     : 5.5 Go
Fichier individus : 1036 Mo


{'coeurs_logiques': 12,
 'coeurs_physiques': 6,
 'memoire_totale_go': 16.0,
 'memoire_libre_go': 5.5,
 'taille_fichier_mo': 1036}

## 2. Convertir

On transforme les données CSV en parquet.

In [3]:
if PARQUET.exists():
    print("Fichier Parquet déjà présent.")
else:
    depart = time.perf_counter()
    df = pd.read_csv(FICHIER)
    print(f"Lecture du CSV : {time.perf_counter() - depart:.1f} s")

    depart = time.perf_counter()
    df.to_parquet(PARQUET, compression="snappy", index=False)
    print(f"Écriture Parquet : {time.perf_counter() - depart:.1f} s")

    del df
    gc.collect()

taille_csv = FICHIER.stat().st_size / 1024**2
taille_parquet = PARQUET.stat().st_size / 1024**2
print(f"\nCSV     : {taille_csv:8.1f} Mo")
print(f"Parquet : {taille_parquet:8.1f} Mo")
print(f"Rapport : ×{taille_csv / taille_parquet:.1f} plus compact")

Lecture du CSV : 42.4 s
Écriture Parquet : 9.8 s

CSV     :   1035.5 Mo
Parquet :    128.8 Mo
Rapport : ×8.0 plus compact


**Question 1.** D'où vient cet écart de taille ? Citez deux mécanismes vus
sur les slides.

*Votre réponse :* …

### Le temps de lecture

In [4]:
_ = pd.read_csv(FICHIER, nrows=10_000)          # lecture à blanc (pour le cache)

print("Lecture intégrale :")
m_csv = mesurer("CSV",     lambda: pd.read_csv(FICHIER))
m_pqt = mesurer("Parquet", lambda: pd.read_parquet(PARQUET))

print(f"\nParquet est ×{m_csv['secondes'] / m_pqt['secondes']:.1f} plus rapide "
      "à lire intégralement.")

Lecture intégrale :
  CSV            43.459 s   pic   3240 Mo
  Parquet         1.567 s   pic   3392 Mo

Parquet est ×27.7 plus rapide à lire intégralement.


**Question 2.** Le fichier Parquet est bien plus petit, mais il faut le
**décompresser** à la lecture. Le gain de temps est-il proportionnel au gain de
taille ? Pourquoi ?

*Votre réponse :* …

## 3. Choisir un codec de compression

Quatre options courantes. Le compromis se joue entre la place occupée, le temps
d'écriture et le temps de lecture.

In [ ]:
# À COMPLÉTER — pour chaque codec, écrivez le fichier, mesurez le temps
# d'écriture, la taille obtenue et le temps de lecture.
# Rassemblez le tout dans un DataFrame `codecs`.
#
# Indice : df.to_parquet(chemin, compression=codec, index=False)
#          compression=None pour l'absence de compression.

df = pd.read_parquet(PARQUET)

resultats = []
for codec in ["snappy", "zstd", "gzip", None]:
    chemin = DOSSIER_DONNEES / f"essai_{codec}.parquet"
    ...

codecs = pd.DataFrame(resultats)
codecs

In [ ]:
del df
gc.collect()

# Ménage : on ne garde que le fichier de référence
for codec in ["snappy", "zstd", "gzip", "None"]:
    chemin = DOSSIER_DONNEES / f"essai_{codec}.parquet"
    if chemin.exists():
        chemin.unlink()

**Question 3.** Quel codec retiendriez-vous pour des données d'archive,
consultées rarement ? Et pour des données de travail, relues plusieurs fois par
jour ? Justifiez avec vos mesures.

*Votre réponse :* …

> Repère mesuré sur un poste de démonstration (2 millions de lignes) :
> `snappy` 42 Mo, `zstd` 32 Mo, `gzip` 31 Mo mais treize fois plus lent à
> écrire, aucune compression 68 Mo mais la lecture la plus rapide.

## 4. La projection de colonnes

Voici le gain principal de Parquet, et de loin. Comme les valeurs d'une même
colonne sont rangées ensemble, en lire deux sur vingt et une revient à lire une
petite fraction du fichier — sans jamais parcourir le reste.

In [ ]:
# À COMPLÉTER — comparez la lecture de toutes les colonnes et celle des deux
# seules colonnes `region` et `age`, en Parquet.
# Puis faites la même chose sur le CSV, pour mesurer ce que le format apporte.
#
# Indices : pd.read_parquet(chemin, columns=[...])
#           pd.read_csv(chemin, usecols=[...])

print("Parquet :")
m_toutes = mesurer("21 colonnes", lambda: ...)
m_deux   = mesurer("2 colonnes",  lambda: ...)

print("\nCSV :")
m_csv_deux = mesurer("2 colonnes", lambda: ...)

print(f"\nParquet, 21 → 2 colonnes : ×{m_toutes['secondes'] / m_deux['secondes']:.0f}")
print(f"CSV → Parquet, à 2 colonnes : ×{m_csv_deux['secondes'] / m_deux['secondes']:.0f}")

**Question 4.** Sur le CSV, lire deux colonnes au lieu de vingt et une
fait-il gagner autant qu'en Parquet ? Pourquoi ?

*Votre réponse :* …

C'est le point à retenir de tout le bloc : en CSV, le moteur doit **parcourir
l'intégralité du fichier** même s'il ne garde que deux colonnes. En Parquet, il
ne lit littéralement pas les octets dont il n'a pas besoin.

## 5. Dans le fichier : groupes de lignes et statistiques

*Les trois cellules suivantes sont une démonstration — regardez, vous n'avez
rien à écrire.*

Un fichier Parquet n'est pas un bloc monolithique : il est découpé en **groupes
de lignes**, et pour chaque groupe et chaque colonne, il conserve des
**statistiques**.

In [5]:
fichier = pq.ParquetFile(PARQUET)
print("Groupes de lignes :", fichier.metadata.num_row_groups)
print("Lignes            :", f"{fichier.metadata.num_rows:,}".replace(",", " "))
print("Colonnes          :", fichier.metadata.num_columns)
print("Créé par          :", fichier.metadata.created_by)

Groupes de lignes : 6
Lignes            : 6 024 000
Colonnes          : 21
Créé par          : parquet-cpp-arrow version 25.0.0


In [6]:
# Le schéma, inscrit dans le fichier : plus aucune inférence de types
fichier.schema_arrow

id_individu: int64
id_menage: large_string
region: large_string
departement: large_string
milieu_residence: large_string
nom: large_string
prenom: large_string
sexe: large_string
age: int64
date_naissance: large_string
lien_chef_menage: large_string
situation_matrimoniale: large_string
niveau_instruction: large_string
sait_lire_ecrire: large_string
situation_activite: large_string
secteur_activite: large_string
nationalite: large_string
type_logement: large_string
nb_pieces: double
source_eau: large_string
electricite: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2602

In [7]:
# Les statistiques du premier groupe de lignes
groupe = fichier.metadata.row_group(0)
print(f"Taille du groupe : {groupe.total_byte_size / 1024**2:.1f} Mo, "
      f"{groupe.num_rows:,} lignes\n".replace(",", " "))

for nom in ["region", "age", "id_individu"]:
    index = fichier.schema_arrow.names.index(nom)
    stats = groupe.column(index).statistics
    print(f"{nom:<14} min={stats.min!s:<12} max={stats.max!s:<14} "
          f"manquants={stats.null_count}")

Taille du groupe : 35.5 Mo  1 048 576 lignes

region         min=  Dakar      max=ziguinchor     manquants=0
age            min=-1           max=999            manquants=0
id_individu    min=1            max=1499999        manquants=0


**Question 5.** Regardez les bornes de la colonne `age`. Que révèlent-elles
sur la qualité des données, et en combien de temps ?

*Votre réponse :* …

Ces statistiques sont **lues sans ouvrir les données** : un profilage instantané
d'un fichier de plusieurs gigaoctets.

## 6. Le saut de blocs — et sa condition

Puisque le fichier connaît les bornes de chaque groupe, un filtre
`region = 'Dakar'` permet d'**ignorer** les groupes dont les bornes excluent
Dakar.

Mais cela n'est possible que si les données sont **regroupées** par cette
colonne. Vérifions.

In [8]:
# Fourni : deux versions du même fichier, l'une triée par région, l'autre non
df = pd.read_parquet(PARQUET)
df["region"] = df["region"].str.strip().str.title()

NON_TRIE = DOSSIER_DONNEES / "individus_nontrie.parquet"
TRIE = DOSSIER_DONNEES / "individus_trie.parquet"

df.to_parquet(NON_TRIE, row_group_size=100_000, compression="snappy", index=False)
df.sort_values("region").to_parquet(TRIE, row_group_size=100_000,
                                    compression="snappy", index=False)
del df
gc.collect()
print("Deux fichiers écrits, en groupes de 100 000 lignes.")

Deux fichiers écrits, en groupes de 100 000 lignes.


In [9]:
# Fourni : combien de groupes de lignes le moteur devra-t-il réellement lire ?
def groupes_concernes(chemin, colonne, valeur):
    fichier = pq.ParquetFile(chemin)
    index = fichier.schema_arrow.names.index(colonne)
    total = fichier.metadata.num_row_groups
    concernes = sum(
        1 for g in range(total)
        if (statistiques := fichier.metadata.row_group(g).column(index).statistics)
        and statistiques.min <= valeur <= statistiques.max
    )
    return concernes, total


for libelle, chemin in [("non trié", NON_TRIE), ("trié par région", TRIE)]:
    concernes, total = groupes_concernes(chemin, "region", "Dakar")
    duree = mesurer(
        libelle,
        lambda ch=chemin: pd.read_parquet(
            ch, columns=["region", "age"], filters=[("region", "==", "Dakar")]),
        afficher=False,
    )["secondes"]
    print(f"{libelle:<18} {concernes:>3} groupes à lire sur {total:<4} "
          f"→ {duree:.3f} s")

non trié            61 groupes à lire sur 61   → 0.643 s
trié par région     14 groupes à lire sur 61   → 0.035 s


**Question 6.** Sur le fichier non trié, pourquoi le moteur doit-il lire
**tous** les groupes, alors même que les statistiques existent ?

*Votre réponse :* …


## 7. Ce qu'il faut retenir

- Parquet est **binaire, colonnaire, compressé, typé** : le format de la donnée
  d'analyse.
- Il divise la taille par 8 à 10, et le temps de lecture intégrale par 5 à 6.
- La **projection de colonnes** apporte bien davantage : lire 2 colonnes sur 21
  peut être trente fois plus rapide — et c'est impossible en CSV.
- Le fichier porte son **schéma** : plus d'inférence de types, plus de dates mal
  lues.
- Les **statistiques par groupe de lignes** permettent de sauter des blocs
  entiers — à condition que les données soient regroupées selon la colonne
  filtrée.

**À compléter :**

- CSV … Mo → Parquet … Mo (×…)
- Lecture intégrale : … s → … s (×…)
- Projection de 21 à 2 colonnes : ×…
- Codec retenu : … parce que …
